# Data Exploration — Clinical QA System
**LJMU MSc ML&AI — Amit Choudhary (1187247)**

This notebook explores the three datasets used in this project:
1. MedQA (USMLE-style MCQs)
2. PubMedQA (biomedical QA with evidence)
3. PubMed Abstracts (retrieval corpus)

Covers: dataset statistics, text length distributions, topic coverage, data quality checks.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import settings

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## 1. MedQA Dataset Analysis

In [ ]:
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

medqa_splits = {}
for split in ['train', 'validation', 'test']:
    p = settings.data_raw_dir / 'medqa' / f'{split}.jsonl'
    if p.exists():
        medqa_splits[split] = load_jsonl(p)
        print(f'MedQA {split}: {len(medqa_splits[split]):,} records')
    else:
        print(f'MedQA {split}: NOT FOUND — run make download-data')

In [ ]:
# Question length distribution
if 'train' in medqa_splits:
    q_lengths = [len(r['question'].split()) for r in medqa_splits['train']]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(q_lengths, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
    axes[0].set_xlabel('Question length (words)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('MedQA Question Length Distribution')
    axes[0].axvline(np.mean(q_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(q_lengths):.0f}')
    axes[0].legend()

    # Answer distribution
    answers = [r.get('answer_idx', r.get('answer', ''))[:1].upper() for r in medqa_splits['train']]
    answer_counts = Counter(answers)
    axes[1].bar(answer_counts.keys(), answer_counts.values(), color=['#2196F3','#4CAF50','#FF9800','#E91E63'])
    axes[1].set_xlabel('Answer choice')
    axes[1].set_ylabel('Count')
    axes[1].set_title('MedQA Answer Distribution (Train)')

    plt.tight_layout()
    plt.savefig('../docs/medqa_stats.png', bbox_inches='tight')
    plt.show()

    print(f'\nMedQA Train Statistics:')
    print(f'  Mean question length: {np.mean(q_lengths):.1f} words')
    print(f'  Median:               {np.median(q_lengths):.1f} words')
    print(f'  Std:                  {np.std(q_lengths):.1f} words')
    print(f'  Min/Max:              {min(q_lengths)} / {max(q_lengths)}')
    print(f'  Answer distribution:  {dict(answer_counts)}')

## 2. PubMedQA Analysis

In [ ]:
pubmedqa_files = list((settings.data_raw_dir / 'pubmedqa').glob('*.jsonl')) if (settings.data_raw_dir / 'pubmedqa').exists() else []
print(f'PubMedQA files found: {[f.name for f in pubmedqa_files]}')

if pubmedqa_files:
    labeled = [f for f in pubmedqa_files if 'labeled' in f.name]
    if labeled:
        records = load_jsonl(labeled[0])
        decisions = Counter(r.get('final_decision', 'unknown') for r in records)
        print(f'  Labeled records: {len(records):,}')
        print(f'  Final decisions: {dict(decisions)}')

        # Context size distribution
        ctx_lengths = [sum(len(c.split()) for c in r.get('context', {}).get('contexts', [])) for r in records]
        answer_lengths = [len(r.get('long_answer', '').split()) for r in records if r.get('long_answer')]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].hist(ctx_lengths, bins=30, color='teal', alpha=0.8)
        axes[0].set_title('Context Size (words)')
        axes[0].set_xlabel('Total context words')

        axes[1].hist(answer_lengths, bins=30, color='coral', alpha=0.8)
        axes[1].set_title('Long Answer Length (words)')
        axes[1].set_xlabel('Words')

        axes[2].bar(decisions.keys(), decisions.values(),
                    color=['#4CAF50', '#F44336', '#9E9E9E'])
        axes[2].set_title('Final Decision Distribution')

        plt.tight_layout()
        plt.savefig('../docs/pubmedqa_stats.png', bbox_inches='tight')
        plt.show()

## 3. PubMed Abstracts Corpus Analysis

In [ ]:
abstracts_path = settings.data_raw_dir / 'pubmed' / 'abstracts.jsonl'
if abstracts_path.exists():
    # Sample 5000 for fast analysis
    abstracts = []
    with open(abstracts_path) as f:
        for i, line in enumerate(f):
            if i >= 5000:
                break
            if line.strip():
                abstracts.append(json.loads(line))

    abs_lengths = [len(r.get('abstract', '').split()) for r in abstracts]
    years = [r.get('year', '') for r in abstracts if r.get('year', '').isdigit()]
    journals = Counter(r.get('journal', '') for r in abstracts).most_common(10)

    print(f'Sample size: {len(abstracts):,}')
    print(f'Avg abstract length: {np.mean(abs_lengths):.0f} words')
    print(f'Year range: {min(years) if years else "N/A"} – {max(years) if years else "N/A"}')
    print(f'\nTop 5 journals:')
    for j, c in journals[:5]:
        print(f'  {c:4d}  {j}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(abs_lengths, bins=40, color='purple', alpha=0.8)
    axes[0].set_title('Abstract Length Distribution')
    axes[0].set_xlabel('Words')
    axes[0].axvline(np.mean(abs_lengths), color='red', linestyle='--')

    if years:
        year_counts = Counter(int(y) for y in years if 2000 <= int(y) <= 2024)
        sorted_years = sorted(year_counts.items())
        axes[1].bar([y for y, _ in sorted_years], [c for _, c in sorted_years],
                    color='slateblue', alpha=0.8)
        axes[1].set_title('Publications by Year')
        axes[1].set_xlabel('Year')
        axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('../docs/pubmed_corpus_stats.png', bbox_inches='tight')
    plt.show()
else:
    print(f'PubMed abstracts not found at {abstracts_path}. Run: make download-data')

## 4. Processed Chunks Analysis

In [ ]:
chunks_path = settings.data_processed_dir / 'pubmed_chunks.jsonl'
if chunks_path.exists():
    chunk_lengths = []
    with open(chunks_path) as f:
        for i, line in enumerate(f):
            if i >= 10000:
                break
            if line.strip():
                r = json.loads(line)
                chunk_lengths.append(len(r.get('text', '').split()))

    print(f'Chunks sampled: {len(chunk_lengths):,}')
    print(f'Mean chunk length: {np.mean(chunk_lengths):.0f} words')
    print(f'P5/P95: {np.percentile(chunk_lengths, 5):.0f} / {np.percentile(chunk_lengths, 95):.0f}')

    plt.figure(figsize=(8, 4))
    plt.hist(chunk_lengths, bins=30, color='darkgreen', alpha=0.8)
    plt.axvline(512, color='red', linestyle='--', label='Target chunk size (512)')
    plt.xlabel('Chunk length (words)')
    plt.ylabel('Count')
    plt.title('Processed Chunk Length Distribution')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../docs/chunk_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print(f'No processed chunks found. Run: make ingest-data')

## 5. Vector Store Statistics

In [ ]:
try:
    from src.data.vectorstore import get_chroma_client, get_or_create_collection
    client = get_chroma_client()
    collection = get_or_create_collection(client)
    count = collection.count()
    print(f'ChromaDB collection: {settings.chroma_collection_name}')
    print(f'Total indexed chunks: {count:,}')
except Exception as e:
    print(f'ChromaDB not available: {e}')